In [1]:
import os
import sys
import argparse

import seaborn as sns
import matplotlib.pyplot as plt

from data import *
from utils import *
from scipy import stats
from datetime import datetime

sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Disentanglement'))
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Accuracy'))
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Attack'))

from DCI import compute_dci, report_dci
from IRS import compute_irs_from_multihot_labels, compute_irs
from MIG import compute_mig
from BA import oh_binary_accuracy
from WB import *

In [2]:
parser = argparse.ArgumentParser(description='Gen-RKM Model')

parser.add_argument('--N', type=int, default=5_000, help='Total # of samples') # 2_000 -> 5_000
parser.add_argument('--mb_size', type=int, default=32, help='Mini-batch size. See utils.py') # 64 -> 32; maybe 48?
parser.add_argument('--h_dim', type=int, default=48, help='Dim of latent vector') # 48 -> 64; maybe 64
parser.add_argument('--capacity', type=int, default=48, help='Capacity of network. See utils.py')

parser.add_argument('--x_fdim', type=int, default=256, help='Input x_fdim. See utils.py')
parser.add_argument('--y_fdim', type=int, default=256, help='Input y_fdim. See utils.py')
parser.add_argument('--z_fdim', type=int, default=64, help='Input z_fdim. See utils.py')

parser.add_argument('--c_accu', type=float, default=50, help='Input weight on recons_error')
parser.add_argument('--start_iter', type=int, default=0, help='Input start_iter for training')

parser.add_argument('--lr', type=float, default=7e-05, help='Input learning rate for optimizer') # 1e-4 -> 7e-5 -> 5e-5
parser.add_argument('--max_epochs', type=int, default=0, help='Input max_epoch for cut-off') # 100 -> 300 -> 500
parser.add_argument('--device', type=str, default='cpu', help='Device type: cuda or cpu')
parser.add_argument('--workers', type=int, default=0, help='# of workers for dataloader')
parser.add_argument('--shuffle', type=bool, default=True, help='shuffle dataset: true or false')

opt, _ = parser.parse_known_args()

h_dim = opt.h_dim

In [3]:
_, xtest, ipVec_dim, nChannels = get_mmcelebahq_dataloader(args=opt)

info 0 5000 1


/Users/arthurclarysse/Desktop/🎓/03 Masterproef CS/VUB-MT-Development/Models/GenRKM-MMCelebHQ-V3/data.py:31: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4483.)
  label = torch.tensor(label, dtype=torch.double).T


info 5001 10001 1


In [25]:
vta = [True, False]
model = '1V'            # 1V, 2V-IS, 2V-IL, 3V
i_val = 10

In [26]:
def kPCA(X, Y, Z, nviews=3, eps=1e-6):
    if nviews == 3:
        a = (X @ X.T + Y @ Y.T + Z @ Z.T) / nviews
    elif nviews == 2:
        a = (X @ X.T + Y @ Y.T) / nviews
    else:
        a = (X @ X.T) / nviews

    B = a.size(0)

    # Centering (same dtype/device as a)
    oneN = torch.ones(B, B, device=a.device, dtype=a.dtype) / B
    a = a - oneN @ a - a @ oneN + oneN @ a @ oneN

    # Force symmetry (numerical)
    a = 0.5 * (a + a.T)

    # Add small jitter to diagonal for conditioning
    a = a + eps * torch.eye(B, device=a.device, dtype=a.dtype)

    # Eigen-decomposition (ascending eigenvalues)
    evals, evecs = torch.linalg.eigh(a)

    # Sort descending
    idx = torch.argsort(evals, descending=True)
    evals = evals[idx]
    evecs = evecs[:, idx]

    # Return top components
    h = evecs[:, :h_dim]
    s = evals  # (B,)

    return h, s

In [27]:
model_1V = torch.load('./out/Final-1V.tar', weights_only=False)
model_2V_IS = torch.load('./out/Final-2V-IS.tar', weights_only=False)
model_2V_IL = torch.load('./out/Final-2V-IL.tar', weights_only=False)
model_3V = torch.load('./out/Final.tar', weights_only=False)

In [28]:
# Single View Networks
net_im_en_1V = model_1V['net_im_en']
net_im_de_1V = model_1V['net_im_de']

# Two View Networks - Image + Sketch
net_im_en_2V_IS = model_2V_IS['net_im_en']
net_sk_en_2V_IS = model_2V_IS['net_sk_en']

net_im_de_2V_IS = model_2V_IS['net_im_de']
net_sk_de_2V_IS = model_2V_IS['net_sk_de']

# Two View Networks - Image + Label
net_im_en_2V_IL = model_2V_IL['net_im_en']
net_la_en_2V_IL = model_2V_IL['net_la_en']

net_im_de_2V_IL = model_2V_IL['net_im_de']
net_la_de_2V_IL = model_2V_IL['net_la_de']

# Three View Networks - Image + Sketch + Label
net_im_en_3V = model_3V['net_im_en']
net_sk_en_3V = model_3V['net_sk_en']
net_la_en_3V = model_3V['net_la_en']

net_im_de_3V = model_3V['net_im_de']
net_sk_de_3V = model_3V['net_sk_de']
net_la_de_3V = model_3V['net_la_de']

In [29]:
net_im_en_1V.eval()
net_im_en_2V_IS.eval()
net_sk_en_2V_IS.eval()
net_im_en_2V_IL.eval()
net_la_en_2V_IL.eval()
net_im_en_3V.eval()
net_sk_en_3V.eval()
net_la_en_3V.eval()

net_im_de_1V.eval()
net_im_de_2V_IS.eval()
net_sk_de_2V_IS.eval()
net_im_de_2V_IL.eval()
net_la_de_2V_IL.eval()
net_im_de_3V.eval()
net_sk_de_3V.eval()
net_la_de_3V.eval()

NetLaDe(
  (fc1): Linear(in_features=64, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=128, out_features=256, bias=True)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc3): Linear(in_features=256, out_features=256, bias=True)
  (bn3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (skip): Linear(in_features=128, out_features=256, bias=True)
  (drop1): Dropout(p=0.1, inplace=False)
  (drop2): Dropout(p=0.1, inplace=False)
  (out): Linear(in_features=256, out_features=40, bias=True)
)

In [30]:
all_data_im, all_data_sk, all_data_la = [], [], []

im_en_1V = []
im_en_2V_IS, sk_en_2V_IS = [], []
im_en_2V_IL, la_en_2V_IL = [], []
im_en_3V, sk_en_3V, la_en_3V = [], [], []

im_de_1V = []
im_de_2V_IS, sk_de_2V_IS = [], []
im_de_2V_IL, la_de_2V_IL = [], []
im_de_3V, sk_de_3V, la_de_3V = [], [], []

im_loss_1V = []
im_loss_2V_IS, sk_loss_2V_IS, loss_2V_IS = [], [], []
im_loss_2V_IL, la_loss_2V_IL, loss_2V_IL = [], [], []
im_loss_3V, sk_loss_3V, la_loss_3V, loss_3V = [], [], [], []

In [31]:
recon_loss1 = torch.nn.MSELoss()
recon_loss2 = torch.nn.MSELoss()
recon_loss3 = torch.nn.BCEWithLogitsLoss()

In [32]:
with torch.no_grad():
    for i, (im, sk, la) in enumerate(xtest):
        if i > i_val * 100:
            break
        
        im = im.to(opt.device)
        sk = sk.to(opt.device)
        la = la.to(opt.device)

        all_data_im.append(im.cpu())
        all_data_sk.append(sk.cpu())
        all_data_la.append(la.cpu())

        # Single View
        im_en_1V.append(net_im_en_1V(im).cpu())

        # Two View - Image + Sketch
        im_en_2V_IS.append(net_im_en_2V_IS(im).cpu())
        sk_en_2V_IS.append(net_sk_en_2V_IS(sk).cpu())

        # Two View - Image + Label
        im_en_2V_IL.append(net_im_en_2V_IL(im).cpu())
        la_en_2V_IL.append(net_la_en_2V_IL(la).cpu())

        # Three View - Image + Sketch + Label
        im_en_3V.append(net_im_en_3V(im).cpu())
        sk_en_3V.append(net_sk_en_3V(sk).cpu())
        la_en_3V.append(net_la_en_3V(la).cpu())

In [33]:
all_data_im = torch.cat(all_data_im, dim=0)[:1000]
all_data_sk = torch.cat(all_data_sk, dim=0)[:1000]
all_data_la = torch.cat(all_data_la, dim=0)[:1000]

im_en_1V = torch.cat(im_en_1V, dim=0)[:1000]
im_en_2V_IS = torch.cat(im_en_2V_IS, dim=0)[:1000]
sk_en_2V_IS = torch.cat(sk_en_2V_IS, dim=0)[:1000]
im_en_2V_IL = torch.cat(im_en_2V_IL, dim=0)[:1000]
la_en_2V_IL = torch.cat(la_en_2V_IL, dim=0)[:1000]
im_en_3V = torch.cat(im_en_3V, dim=0)[:1000]
sk_en_3V = torch.cat(sk_en_3V, dim=0)[:1000]
la_en_3V = torch.cat(la_en_3V, dim=0)[:1000]

In [34]:
with torch.no_grad():
    for i in range(i_val):
        start, stop = i * 100, (i + 1) * 100

        # Get latent representations (current batch of 100 samples)
        h_1V, _ = kPCA(im_en_1V[start:stop], None, None, nviews=1)
        h_2V_IS, _ = kPCA(im_en_2V_IS[start:stop], sk_en_2V_IS[start:stop], None, nviews=2)
        h_2V_IL, _ = kPCA(im_en_2V_IL[start:stop], la_en_2V_IL[start:stop], None, nviews=2)
        h_3V, _ = kPCA(im_en_3V[start:stop], sk_en_3V[start:stop], la_en_3V[start:stop], nviews=3)

        # Calculate U // All views
        U_1V = im_en_1V[start:stop].T @ h_1V
        U_2V_IS = im_en_2V_IS[start:stop].T @ h_2V_IS
        U_2V_IL = im_en_2V_IL[start:stop].T @ h_2V_IL
        U_3V = im_en_3V[start:stop].T @ h_3V

        # Calculate V // 2V-IS and 3V only
        V_2V_IS = sk_en_2V_IS[start:stop].T @ h_2V_IS
        V_3V = sk_en_3V[start:stop].T @ h_3V

        # Calculate W // 2V-IL and 3V only
        W_2V_IL = la_en_2V_IL[start:stop].T @ h_2V_IL
        W_3V = la_en_3V[start:stop].T @ h_3V

        # Reconstruct images, sketches, labels from latent representations
        im_1V_tilde = net_im_de_1V(h_1V @ U_1V.T)
        im_2V_IS_tilde = net_im_de_2V_IS(h_2V_IS @ U_2V_IS.T)
        im_2V_IL_tilde = net_im_de_2V_IL(h_2V_IL @ U_2V_IL.T)
        im_3V_tilde = net_im_de_3V(h_3V @ U_3V.T)

        sk_2V_IS_tilde = net_sk_de_2V_IS(h_2V_IS @ V_2V_IS.T)
        sk_3V_tilde = net_sk_de_3V(h_3V @ V_3V.T)

        la_2V_IL_tilde = net_la_de_2V_IL(h_2V_IL @ W_2V_IL.T)
        la_3V_tilde = net_la_de_3V(h_3V @ W_3V.T)

        # Calculate losses
        im_true = all_data_im[start:stop]
        sk_true = all_data_sk[start:stop]
        la_true = all_data_la[start:stop]

        im_loss_1V.append(recon_loss1(im_1V_tilde.view(im_1V_tilde.size(0), -1), im_true.view(im_true.size(0), -1)))
        im_loss_2V_IS.append(recon_loss1(im_2V_IS_tilde.view(im_2V_IS_tilde.size(0), -1), im_true.view(im_true.size(0), -1)))
        im_loss_2V_IL.append(recon_loss1(im_2V_IL_tilde.view(im_2V_IL_tilde.size(0), -1), im_true.view(im_true.size(0), -1)))
        im_loss_3V.append(recon_loss1(im_3V_tilde.view(im_3V_tilde.size(0), -1), im_true.view(im_true.size(0), -1)))

        sk_loss_2V_IS.append(recon_loss2(sk_2V_IS_tilde.view(sk_2V_IS_tilde.size(0), -1), sk_true.view(sk_true.size(0), -1)))
        sk_loss_3V.append(recon_loss2(sk_3V_tilde.view(sk_3V_tilde.size(0), -1), sk_true.view(sk_true.size(0), -1)))

        la_loss_2V_IL.append(recon_loss3(la_2V_IL_tilde.view(la_2V_IL_tilde.size(0), -1), la_true.view(la_true.size(0), -1)))
        la_loss_3V.append(recon_loss3(la_3V_tilde.view(la_3V_tilde.size(0), -1), la_true.view(la_true.size(0), -1)))

        # Store some samples for visualization
        if i == 0:
            im_de_1V.append(im_1V_tilde.cpu()[:10])
            im_de_2V_IS.append(im_2V_IS_tilde.cpu()[:10])
            im_de_2V_IL.append(im_2V_IL_tilde.cpu()[:10])
            im_de_3V.append(im_3V_tilde.cpu()[:10])
            sk_de_2V_IS.append(sk_2V_IS_tilde.cpu()[:10])
            sk_de_3V.append(sk_3V_tilde.cpu()[:10])
            la_de_2V_IL.append(la_2V_IL_tilde.cpu()[:10])
            la_de_3V.append(la_3V_tilde.cpu()[:10])

        print(f'Batch {i + 1}/{i_val} processed.')

Batch 1/10 processed.
Batch 2/10 processed.
Batch 3/10 processed.
Batch 4/10 processed.
Batch 5/10 processed.
Batch 6/10 processed.
Batch 7/10 processed.
Batch 8/10 processed.
Batch 9/10 processed.
Batch 10/10 processed.


## Attack

In [35]:
epsilon = 0.2

data_im = all_data_im.cpu()
data_sk = all_data_sk.cpu()
data_la = all_data_la.cpu()

In [36]:
FGSM_3V_I = WB_Attack(net_in1=net_im_en_3V, net_out1=net_im_de_3V, net_in2=net_sk_en_3V, net_out2=net_sk_de_3V, net_in3=net_la_en_3V, net_out3=net_la_de_3V, kPCA=kPCA, opt=opt, vta=[True, False])
FGSM_3V_S = WB_Attack(net_in1=net_im_en_3V, net_out1=net_im_de_3V, net_in2=net_sk_en_3V, net_out2=net_sk_de_3V, net_in3=net_la_en_3V, net_out3=net_la_de_3V, kPCA=kPCA, opt=opt, vta=[False, True])
FGSM_3V_IS = WB_Attack(net_in1=net_im_en_3V, net_out1=net_im_de_3V, net_in2=net_sk_en_3V, net_out2=net_sk_de_3V, net_in3=net_la_en_3V, net_out3=net_la_de_3V, kPCA=kPCA, opt=opt, vta=[True, True])

FGSM_2V_IS_I = WB_Attack(net_in1=net_im_en_2V_IS, net_out1=net_im_de_2V_IS, net_in2=net_sk_en_2V_IS, net_out2=net_sk_de_2V_IS, net_in3=None, net_out3=None, kPCA=kPCA, opt=opt, vta=[True, False])
FGSM_2V_IS_S = WB_Attack(net_in1=net_im_en_2V_IS, net_out1=net_im_de_2V_IS, net_in2=net_sk_en_2V_IS, net_out2=net_sk_de_2V_IS, net_in3=None, net_out3=None, kPCA=kPCA, opt=opt, vta=[False, True])
FGSM_2V_IS_IS = WB_Attack(net_in1=net_im_en_2V_IS, net_out1=net_im_de_2V_IS, net_in2=net_sk_en_2V_IS, net_out2=net_sk_de_2V_IS, net_in3=None, net_out3=None, kPCA=kPCA, opt=opt, vta=[True, True])

FGSM_2V_IL_I = WB_Attack(net_in1=net_im_en_2V_IL, net_out1=net_im_de_2V_IL, net_in2=None, net_out2=None, net_in3=net_la_en_2V_IL, net_out3=net_la_de_2V_IL, kPCA=kPCA, opt=opt, vta=[True, False])
FGSM_1V_I = WB_Attack(net_in1=net_im_en_1V, net_out1=net_im_de_1V, net_in2=None, net_out2=None, net_in3=None, net_out3=None, kPCA=kPCA, opt=opt, vta=[True, False])

In [37]:
if model == '3V':
    if vta[0] and vta[1]:
        attack = FGSM_3V_IS
    elif vta[0] and not vta[1]:
        attack = FGSM_3V_I
    elif not vta[0] and vta[1]:
        attack = FGSM_3V_S
    else:
        raise ValueError('Invalid VTA choice. Choose from [T, T], [T, F], or [F, T].')

elif model == '2V-IS':
    if vta[0] and vta[1]:
        attack = FGSM_2V_IS_IS
    elif vta[0] and not vta[1]:
        attack = FGSM_2V_IS_I
    elif not vta[0] and vta[1]:
        attack = FGSM_2V_IS_S
    else:
        raise ValueError('Invalid VTA choice. Choose from [T, T] or [T, F].')

elif model == '2V-IL':
    attack = FGSM_2V_IL_I

elif model == '1V':
    attack = FGSM_1V_I
    
else:
    raise ValueError('Invalid model choice. Choose from: 1V, 2V-IS, 2V-IL, 3V.')

In [38]:
adv_batches_fgsm_I, adv_batches_fgsm_S, adv_batches_bim_I, adv_batches_bim_S = [], [], [], []

print(f'---- Creating Adversarial Batches for Model: {model} with VTA: {vta} ----')
for i in range(i_val):
    start, stop = i * 100, (i + 1) * 100
    batch_im = data_im[start:stop]
    batch_sk = data_sk[start:stop]
    batch_la = data_la[start:stop]

    if model == '2V-IL':
        batch_sk = None
    elif model == '2V-IS':
        batch_la = None
    elif model == '1V':
        batch_sk = None
        batch_la = None


    print(f' > Processing batch {i + 1}/{i_val}...')

    if vta[0] and vta[1]:
        adv_batch_fgsm_I, adv_batch_fgsm_S, _, _ = attack.create_adversarial_batch_fgsm(batch_im, batch_sk, batch_la, epsilon=epsilon)
        print(f'   > FGSM adversarial batch created.')

        adv_batch_bim_I, adv_batch_bim_S, _, _ = attack.create_adversarial_batch_bim(batch_im, batch_sk, batch_la, epsilon=epsilon, alpha=0.1, num_iterations=10)
        print(f'   > BIM adversarial batch created.\n')

        adv_batches_fgsm_I.append(adv_batch_fgsm_I.cpu())
        adv_batches_fgsm_S.append(adv_batch_fgsm_S.cpu())
        adv_batches_bim_I.append(adv_batch_bim_I.cpu())
        adv_batches_bim_S.append(adv_batch_bim_S.cpu())

    elif vta[0] and not vta[1]:
        adv_batch_fgsm_I, _ = attack.create_adversarial_batch_fgsm(batch_im, batch_sk, batch_la, epsilon=epsilon)
        print(f'   > FGSM adversarial batch created.')

        adv_batch_bim_I, _ = attack.create_adversarial_batch_bim(batch_im, batch_sk, batch_la, epsilon=epsilon, alpha=0.1, num_iterations=10)
        print(f'   > BIM adversarial batch created.\n')

        adv_batches_fgsm_I.append(adv_batch_fgsm_I.cpu())
        adv_batches_bim_I.append(adv_batch_bim_I.cpu())

    elif not vta[0] and vta[1]:
        adv_batch_fgsm_S, _ = attack.create_adversarial_batch_fgsm(batch_im, batch_sk, batch_la, epsilon=epsilon)
        print(f'   > FGSM adversarial batch created.')

        adv_batch_bim_S, _, = attack.create_adversarial_batch_bim(batch_im, batch_sk, batch_la, epsilon=epsilon, alpha=0.1, num_iterations=10)
        print(f'   > BIM adversarial batch created.\n')

        adv_batches_fgsm_S.append(adv_batch_fgsm_S.cpu())
        adv_batches_bim_S.append(adv_batch_bim_S.cpu())

    else:
        adv_batch_fgsm_I, _ = attack.create_adversarial_batch_fgsm(batch_im, batch_sk, batch_la, epsilon=epsilon)
        print(f'   > FGSM adversarial batch created.')

        adv_batch_bim_I, _ = attack.create_adversarial_batch_bim(batch_im, batch_sk, batch_la, epsilon=epsilon, alpha=0.1, num_iterations=10)
        print(f'   > BIM adversarial batch created.\n')

        adv_batches_fgsm_I.append(adv_batch_fgsm_I.cpu())
        adv_batches_bim_I.append(adv_batch_bim_I.cpu())

---- Creating Adversarial Batches for Model: 1V with VTA: [True, False] ----
 > Processing batch 1/10...
   > FGSM adversarial batch created.
   > BIM adversarial batch created.

 > Processing batch 2/10...
   > FGSM adversarial batch created.
   > BIM adversarial batch created.

 > Processing batch 3/10...
   > FGSM adversarial batch created.
   > BIM adversarial batch created.

 > Processing batch 4/10...
   > FGSM adversarial batch created.
   > BIM adversarial batch created.

 > Processing batch 5/10...
   > FGSM adversarial batch created.
   > BIM adversarial batch created.

 > Processing batch 6/10...
   > FGSM adversarial batch created.
   > BIM adversarial batch created.

 > Processing batch 7/10...
   > FGSM adversarial batch created.
   > BIM adversarial batch created.

 > Processing batch 8/10...
   > FGSM adversarial batch created.
   > BIM adversarial batch created.

 > Processing batch 9/10...
   > FGSM adversarial batch created.
   > BIM adversarial batch created.

 > Pr

In [39]:
print(f'---- Running Adversarial Batches through the Model ----')
fgsm_xs_tilde, fgsm_ys_tilde, fgsm_zs_tilde = [], [], []
bim_xs_tilde, bim_ys_tilde, bim_zs_tilde = [], [], []

with torch.no_grad():
    for i in range(i_val):
        print(f' > Processing adversarial batch {i + 1}/{i_val}...')
        data_im_batch = data_im[i * 100:(i + 1) * 100].to(opt.device)
        data_sk_batch = data_sk[i * 100:(i + 1) * 100].to(opt.device)
        data_la_batch = data_la[i * 100:(i + 1) * 100].to(opt.device)

        if model == '2V-IL':
            data_sk_batch = None
        elif model == '2V-IS':
            data_la_batch = None
        elif model == '1V':
            data_sk_batch = None
            data_la_batch = None
        
        if vta[0] and vta[1]:
            adv_batch_fgsm_I = adv_batches_fgsm_I[i].to(opt.device)
            adv_batch_bim_I = adv_batches_bim_I[i].to(opt.device)
            adv_batch_fgsm_S = adv_batches_fgsm_S[i].to(opt.device)
            adv_batch_bim_S = adv_batches_bim_S[i].to(opt.device)

            fgsm_x_tilde, fgsm_y_tilde, fgsm_z_tilde = attack.run_model_batch(adv_batch_fgsm_I, adv_batch_fgsm_S, data_la_batch)
            print(f'   > FGSM <TT> adversarial batch processed.')
            bim_x_tilde, bim_y_tilde, bim_z_tilde = attack.run_model_batch(adv_batch_bim_I, adv_batch_bim_S, data_la_batch)
            print(f'   > BIM <TT> adversarial batch processed.\n')

        elif vta[0] and not vta[1]:
            adv_batch_fgsm_I = adv_batches_fgsm_I[i].to(opt.device)
            adv_batch_bim_I = adv_batches_bim_I[i].to(opt.device)

            fgsm_x_tilde, fgsm_y_tilde, fgsm_z_tilde = attack.run_model_batch(adv_batch_fgsm_I, data_sk_batch, data_la_batch)
            print(f'   > FGSM <TF> adversarial batch processed.')
            bim_x_tilde, bim_y_tilde, bim_z_tilde = attack.run_model_batch(adv_batch_bim_I, data_sk_batch, data_la_batch)
            print(f'   > BIM <TF> adversarial batch processed.\n')

        elif not vta[0] and vta[1]:
            adv_batch_fgsm_S = adv_batches_fgsm_S[i].to(opt.device)
            adv_batch_bim_S = adv_batches_bim_S[i].to(opt.device)
                
            fgsm_x_tilde, fgsm_y_tilde, fgsm_z_tilde = attack.run_model_batch(data_im_batch, adv_batch_fgsm_S, data_la_batch)
            print(f'   > FGSM <FT> adversarial batch processed.')
            bim_x_tilde, bim_y_tilde, bim_z_tilde = attack.run_model_batch(data_im_batch, adv_batch_bim_S, data_la_batch)
            print(f'   > BIM <FT> adversarial batch processed.\n')

        else:
            raise ValueError('Invalid VTA choice. Choose from [T, T], [T, F], or [F, T].')

        fgsm_xs_tilde.append(fgsm_x_tilde.cpu())
        fgsm_ys_tilde.append(fgsm_y_tilde.cpu()) if fgsm_y_tilde is not None else None
        fgsm_zs_tilde.append(fgsm_z_tilde.cpu()) if fgsm_z_tilde is not None else None
        bim_xs_tilde.append(bim_x_tilde.cpu())
        bim_ys_tilde.append(bim_y_tilde.cpu()) if bim_y_tilde is not None else None
        bim_zs_tilde.append(bim_z_tilde.cpu()) if bim_z_tilde is not None else None

---- Running Adversarial Batches through the Model ----
 > Processing adversarial batch 1/10...
   > FGSM <TF> adversarial batch processed.
   > BIM <TF> adversarial batch processed.

 > Processing adversarial batch 2/10...
   > FGSM <TF> adversarial batch processed.
   > BIM <TF> adversarial batch processed.

 > Processing adversarial batch 3/10...
   > FGSM <TF> adversarial batch processed.
   > BIM <TF> adversarial batch processed.

 > Processing adversarial batch 4/10...
   > FGSM <TF> adversarial batch processed.
   > BIM <TF> adversarial batch processed.

 > Processing adversarial batch 5/10...
   > FGSM <TF> adversarial batch processed.
   > BIM <TF> adversarial batch processed.

 > Processing adversarial batch 6/10...
   > FGSM <TF> adversarial batch processed.
   > BIM <TF> adversarial batch processed.

 > Processing adversarial batch 7/10...
   > FGSM <TF> adversarial batch processed.
   > BIM <TF> adversarial batch processed.

 > Processing adversarial batch 8/10...
   > FGS

## Attack Loss

In [40]:
im_loss = torch.nn.MSELoss()
sk_loss = torch.nn.MSELoss()
la_loss = torch.nn.BCEWithLogitsLoss()

B = data_im.size(0)

In [41]:
if model == '3V':
    clean_im_loss = im_loss_3V
    clean_sk_loss = sk_loss_3V
    clean_la_loss = la_loss_3V

elif model == '2V-IS':
    clean_im_loss = im_loss_2V_IS
    clean_sk_loss = sk_loss_2V_IS
    clean_la_loss = None

elif model == '2V-IL':
    clean_im_loss = im_loss_2V_IL
    clean_sk_loss = None
    clean_la_loss = la_loss_2V_IL

elif model == '1V':
    clean_im_loss = im_loss_1V
    clean_sk_loss = None
    clean_la_loss = None

clean_im_loss = np.array(clean_im_loss)
clean_sk_loss = np.array(clean_sk_loss)
clean_la_loss = np.array(clean_la_loss)

In [42]:
fgsm_losses_im, bim_losses_im = [], []
fgsm_losses_sk, bim_losses_sk = [], []
fgsm_losses_la, bim_losses_la = [], []

batch_size = 100

print(f'---- Calculating Losses for Adversarial Batches ----')
for i in range(i_val):
    start, stop = i * batch_size, (i + 1) * batch_size
    batch_im = data_im[start:stop]

    batch_sk = data_sk[start:stop] if data_sk is not None else None
    batch_la = data_la[start:stop] if data_la is not None else None

    print(f' > Calculating losses for adversarial batch {i + 1}/{i_val}...')

    # Image loss is always available
    fgsm_loss_im = im_loss(fgsm_xs_tilde[i], batch_im)
    bim_loss_im  = im_loss(bim_xs_tilde[i], batch_im)

    fgsm_losses_im.append(fgsm_loss_im.item() if hasattr(fgsm_loss_im, "item") else fgsm_loss_im)
    bim_losses_im.append(bim_loss_im.item() if hasattr(bim_loss_im, "item") else bim_loss_im)

    print(f'   > Image losses calculated.')

    # Sketch loss
    if batch_sk is not None and len(fgsm_ys_tilde) > 0 and len(bim_ys_tilde) > 0:
        fgsm_loss_sk = sk_loss(fgsm_ys_tilde[i], batch_sk)
        bim_loss_sk  = sk_loss(bim_ys_tilde[i], batch_sk)

        fgsm_losses_sk.append(fgsm_loss_sk.item() if hasattr(fgsm_loss_sk, "item") else fgsm_loss_sk)
        bim_losses_sk.append(bim_loss_sk.item() if hasattr(bim_loss_sk, "item") else bim_loss_sk)
        print(f'   > Sketch losses calculated.')
    else:
        fgsm_losses_sk.append(None)
        bim_losses_sk.append(None)

    # Label loss
    if batch_la is not None and len(fgsm_zs_tilde) > 0 and len(bim_zs_tilde) > 0:
        fgsm_loss_la = la_loss(fgsm_zs_tilde[i], batch_la)
        bim_loss_la  = la_loss(bim_zs_tilde[i], batch_la)

        fgsm_losses_la.append(fgsm_loss_la.item() if hasattr(fgsm_loss_la, "item") else fgsm_loss_la)
        bim_losses_la.append(bim_loss_la.item() if hasattr(bim_loss_la, "item") else bim_loss_la)
        print(f'   > Label losses calculated.\n')
    else:
        fgsm_losses_la.append(None)
        bim_losses_la.append(None)
        print()

fgsm_losses_im = np.array(fgsm_losses_im, dtype=float)
bim_losses_im  = np.array(bim_losses_im, dtype=float)

fgsm_losses_sk = np.array(fgsm_losses_sk, dtype=object)
bim_losses_sk  = np.array(bim_losses_sk, dtype=object)

fgsm_losses_la = np.array(fgsm_losses_la, dtype=object)
bim_losses_la  = np.array(bim_losses_la, dtype=object)

---- Calculating Losses for Adversarial Batches ----
 > Calculating losses for adversarial batch 1/10...
   > Image losses calculated.

 > Calculating losses for adversarial batch 2/10...
   > Image losses calculated.

 > Calculating losses for adversarial batch 3/10...
   > Image losses calculated.

 > Calculating losses for adversarial batch 4/10...
   > Image losses calculated.

 > Calculating losses for adversarial batch 5/10...
   > Image losses calculated.

 > Calculating losses for adversarial batch 6/10...
   > Image losses calculated.

 > Calculating losses for adversarial batch 7/10...
   > Image losses calculated.

 > Calculating losses for adversarial batch 8/10...
   > Image losses calculated.

 > Calculating losses for adversarial batch 9/10...
   > Image losses calculated.

 > Calculating losses for adversarial batch 10/10...
   > Image losses calculated.



In [43]:
print(f'---- Evaluation Results for Model: {model} with VTA: {vta} ----')
print(f'\tClean\t\tFGSM\t\tBIM')

# Image losses are always available
print(
    f'IM\t{round(np.nanmean(clean_im_loss), 4)}\t\t'
    f'{round(np.nanmean(fgsm_losses_im), 4)}\t\t'
    f'{round(np.nanmean(bim_losses_im), 4)}'
)

# Sketch losses
if clean_sk_loss is not None and fgsm_losses_sk is not None and bim_losses_sk is not None:
    clean_sk_vals = np.array(clean_sk_loss, dtype=float)
    fgsm_sk_vals = np.array([x if x is not None else np.nan for x in fgsm_losses_sk], dtype=float)
    bim_sk_vals  = np.array([x if x is not None else np.nan for x in bim_losses_sk], dtype=float)

    if not np.all(np.isnan(clean_sk_vals)):
        print(
            f'SK\t{round(np.nanmean(clean_sk_vals), 4)}\t\t'
            f'{round(np.nanmean(fgsm_sk_vals), 4)}\t\t'
            f'{round(np.nanmean(bim_sk_vals), 4)}'
        )

# Label losses
if clean_la_loss is not None and fgsm_losses_la is not None and bim_losses_la is not None:
    clean_la_vals = np.array(clean_la_loss, dtype=float)
    fgsm_la_vals = np.array([x if x is not None else np.nan for x in fgsm_losses_la], dtype=float)
    bim_la_vals  = np.array([x if x is not None else np.nan for x in bim_losses_la], dtype=float)

    if not np.all(np.isnan(clean_la_vals)):
        print(
            f'LA\t{round(np.nanmean(clean_la_vals), 4)}\t\t'
            f'{round(np.nanmean(fgsm_la_vals), 4)}\t\t'
            f'{round(np.nanmean(bim_la_vals), 4)}'
        )

---- Evaluation Results for Model: 1V with VTA: [True, False] ----
	Clean		FGSM		BIM
IM	0.0119		0.127		0.0739


In [44]:
# test for significance of differences using paired t-tests
C_F_im_test = stats.ttest_rel(clean_im_loss, fgsm_losses_im)
C_B_im_test = stats.ttest_rel(clean_im_loss, bim_losses_im)
F_B_im_test = stats.ttest_rel(fgsm_losses_im, bim_losses_im)

print(f'---- Paired t-test results for Image Loss: ----')
print(f' > Clean vs FGSM: t-statistic = {C_F_im_test.statistic:.4f}, p-value = {C_F_im_test.pvalue}')
print(f' > Clean vs BIM: t-statistic = {C_B_im_test.statistic:.4f}, p-value = {C_B_im_test.pvalue}')
print(f' > FGSM vs BIM: t-statistic = {F_B_im_test.statistic:.4f}, p-value = {F_B_im_test.pvalue}')

---- Paired t-test results for Image Loss: ----
 > Clean vs FGSM: t-statistic = -155.4697, p-value = 9.580730080656623e-17
 > Clean vs BIM: t-statistic = -90.1954, p-value = 1.2830660794380497e-14
 > FGSM vs BIM: t-statistic = 153.0672, p-value = 1.1021704712880123e-16


In [45]:
# Save the losses to a CSV file
results_df = pd.DataFrame({
    'model': [model] * i_val,
    'image_attack': [vta[0]] * i_val,
    'sketch_attack': [vta[1]] * i_val,

    'Clean_IM_Loss': clean_im_loss,
    'FGSM_IM_Loss': fgsm_losses_im,
    'BIM_IM_Loss': bim_losses_im,

    'Clean_SK_Loss': clean_sk_loss if clean_sk_loss is not None else [None] * i_val,
    'FGSM_SK_Loss': fgsm_losses_sk if fgsm_losses_sk is not None else [None] * i_val,
    'BIM_SK_Loss': bim_losses_sk if bim_losses_sk is not None else [None] * i_val,

    'Clean_LA_Loss': clean_la_loss if clean_la_loss is not None else [None] * i_val,
    'FGSM_LA_Loss': fgsm_losses_la if fgsm_losses_la is not None else [None] * i_val,
    'BIM_LA_Loss': bim_losses_la if bim_losses_la is not None else [None] * i_val
})

# Append to file if it exists, otherwise create new
results_file = 'attack_results.csv'
if os.path.exists(results_file):
    results_df.to_csv(results_file, mode='a', header=False, index=False)
else:
    results_df.to_csv(results_file, index=False)